# 5 · Auto-discovery and JSON schema

Two production features:

1. **Auto-discovery** — `registry.discover("myapp.nodes")` imports every module in a package, triggering all `@registry.node` decorators in one call.
2. **JSON schema** — `serialize_registry(registry)` dumps the registry to JSON so a frontend can render node palettes and input widgets.

Nodes are **versioned** (`base_id@version`). Registering `echo@2` doesn't remove `echo@1`; it just becomes the latest.

In [1]:
from typing import Annotated

from conductor import NodeRegistry
from conductor.registry.schema import serialize_registry
from conductor.widgets import Output, Text

registry = NodeRegistry()


@registry.node("echo", version=1, name="Echo", description="Echoes input")
def echo_v1(text: Annotated[str, Text(label="Input")]) -> Annotated[str, Output(label="Output")]:
    return text


@registry.node("echo", version=2, name="Echo v2", description="Echo with prefix support")
def echo_v2(
    text: Annotated[str, Text(label="Input")],
    prefix: Annotated[str, Text(label="Prefix")] = "",
) -> Annotated[str, Output(label="Output")]:
    return f"{prefix}{text}" 

## Versioning

Old versions stay registered but are flagged as deprecated.

In [2]:
print("echo@1 deprecated? ", registry.is_deprecated("echo@1"))
print("echo@2 deprecated? ", registry.is_deprecated("echo@2"))
print("latest echo:        ", registry.get_latest("echo").id)

echo@1 deprecated?  True
echo@2 deprecated?  False
latest echo:         echo@2


## Auto-discovery (concept)

If you have a package like:

```
myapp/
  nodes/
    __init__.py
    text.py     # contains @registry.node("echo", ...) etc.
    math.py     # contains @registry.node("add", ...) etc.
```

Register everything with one call:

```python
registry.discover("myapp.nodes")
```

It walks the package and imports every module — each import runs the decorators and populates the registry.

## JSON schema for a frontend

`serialize_registry()` returns a `list[dict]` — one entry per node version, with everything a frontend needs to render it.

In [3]:
import json

schema = serialize_registry(registry)
print(json.dumps(schema, indent=2))

[
  {
    "id": "echo@1",
    "base_id": "echo",
    "version": 1,
    "name": "Echo",
    "description": "Echoes input",
    "tags": [],
    "category": "io",
    "inputs": [
      {
        "name": "text",
        "type": "str",
        "label": "Input",
        "description": null,
        "widget": "text",
        "default": null,
        "optional": false,
        "disable_handle": false
      }
    ],
    "outputs": [
      {
        "name": "result",
        "type": "str",
        "label": "Output",
        "description": null,
        "optional": false,
        "download": false,
        "filename": null
      }
    ],
    "width": null,
    "deprecated": true,
    "latest_version": 2,
    "docs": null
  },
  {
    "id": "echo@2",
    "base_id": "echo",
    "version": 2,
    "name": "Echo v2",
    "description": "Echo with prefix support",
    "tags": [],
    "category": "io",
    "inputs": [
      {
        "name": "text",
        "type": "str",
        "label": "Input",
     

## Schema summary

In [4]:
print(f"Total node versions: {len(schema)}\n")
for node in schema:
    status = "DEPRECATED" if node["deprecated"] else "current"
    print(f"  {node['id']}: {node['name']} ({status})")
    for inp in node["inputs"]:
        opt = " (optional)" if inp["optional"] else ""
        print(f"    in:  {inp['name']}: {inp['type']} [{inp['widget']}]{opt}")
    for out in node["outputs"]:
        print(f"    out: {out['name']}: {out['type']}")
    print()

Total node versions: 2

  echo@1: Echo (DEPRECATED)
    in:  text: str [text]
    out: result: str

  echo@2: Echo v2 (current)
    in:  text: str [text]
    in:  prefix: str [text] (optional)
    out: result: str

